<a href="https://colab.research.google.com/github/thetireddude/on-policy-distillation-lab/blob/main/Thinking_Machines_OPD.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

GPU Check

In [242]:
!nvidia-smi

Fri Jul  3 03:50:11 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   31C    P0             62W /  400W |   21450MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

Install Libraries

In [243]:
!pip install -U transformers accelerate datasets peft bitsandbytes sentencepiece tqdm

Imports Setup

In [244]:
import os
import gc
import math
import json
import random
import torch
import torch.nn.functional as F

from tqdm.auto import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

In [245]:
import os
from google.colab import userdata

# Automatically retrieves your token and authenticates the environment
os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')

In [246]:
!hf auth login

User is already logged in. Use `hf auth login --force` to force re-login.


In [247]:
!hf auth whoami

✓ Logged in
  user: tiredk


In [248]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Configuration

In [249]:
import os

SAVE_DIR = "/content/drive/MyDrive/algoverse_opd_experiment/opd_seed789_grp4_pps4_tok128"

os.makedirs(SAVE_DIR, exist_ok=True)

print(SAVE_DIR)

/content/drive/MyDrive/algoverse_opd_experiment/opd_seed789_grp4_pps4_tok128


In [250]:
SEED = 789
random.seed(SEED)
torch.manual_seed(SEED)

STUDENT_MODEL = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
TEACHER_MODEL = "Qwen/Qwen2.5-Coder-7B-Instruct"

PROMPTS_PER_STEP = 4
GROUP_SIZE = 4
MAX_NEW_TOKENS = 128   # use 256 after smoke test works
TEMPERATURE = 1.0
TOP_P = 0.95

LORA_RANK = 64
LORA_ALPHA = 128
LR = 1e-4
EPOCHS = 1

KL_PENALTY_COEF = 1.0
ACCUM_STEPS = 1
CHECKPOINT_EVERY = 82

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

torch.backends.cuda.matmul.allow_tf32 = True

print("Device:", DEVICE)

Device: cuda


GPU Monitor

In [251]:
import subprocess, threading, time

def gpu_monitor(interval=30):
    while True:
        result = subprocess.run(
            [
                "nvidia-smi",
                "--query-gpu=utilization.gpu,memory.used,memory.total",
                "--format=csv,noheader,nounits",
            ],
            capture_output=True,
            text=True,
        )
        print(f"[GPU] {result.stdout.strip()}")
        time.sleep(interval)

threading.Thread(target=gpu_monitor, daemon=True).start()

Load HumanEval Dataset Prompts

In [252]:
dataset = load_dataset("openai/openai_humaneval", split="test")
print(dataset)
print(dataset[0].keys())
print(dataset[0]["prompt"][:500])

[GPU] 0, 21450, 81920
[GPU] 0, 21450, 81920
Dataset({
    features: ['task_id', 'prompt', 'canonical_solution', 'test', 'entry_point'],
    num_rows: 164
})
dict_keys(['task_id', 'prompt', 'canonical_solution', 'test', 'entry_point'])
from typing import List


def has_close_elements(numbers: List[float], threshold: float) -> bool:
    """ Check if in given list of numbers, are any two numbers closer to each other than
    given threshold.
    >>> has_close_elements([1.0, 2.0, 3.0], 0.5)
    False
    >>> has_close_elements([1.0, 2.8, 3.0, 4.0, 5.0, 2.0], 0.3)
    True
    """



Build Prompt Format

In [253]:
def make_prompt(problem):
    return (
        "You are an expert Python programmer.\n"
        "Complete the following Python function.\n"
        "Return only valid Python code. Do not explain.\n\n"
        f"{problem['prompt']}"
    )

prompts = [make_prompt(x) for x in dataset]
task_ids = [x["task_id"] for x in dataset]

print(prompts[0])

You are an expert Python programmer.
Complete the following Python function.
Return only valid Python code. Do not explain.

from typing import List


def has_close_elements(numbers: List[float], threshold: float) -> bool:
    """ Check if in given list of numbers, are any two numbers closer to each other than
    given threshold.
    >>> has_close_elements([1.0, 2.0, 3.0], 0.5)
    False
    >>> has_close_elements([1.0, 2.8, 3.0, 4.0, 5.0, 2.0], 0.3)
    True
    """



Load Tokenizer

In [254]:
tokenizer = AutoTokenizer.from_pretrained(STUDENT_MODEL, trust_remote_code=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "left"

print("pad token:", tokenizer.pad_token)
print("eos token:", tokenizer.eos_token)

pad token: <|endoftext|>
eos token: <|im_end|>


Load Student Model in 4-bit + LoRA

In [255]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

student = AutoModelForCausalLM.from_pretrained(
    STUDENT_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
)

student.config.use_cache = False
student = prepare_model_for_kbit_training(student)

lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

student = get_peft_model(student, lora_config)
student.print_trainable_parameters()

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


[GPU] 27, 21450, 81920
[GPU] 26, 21450, 81920
trainable params: 73,859,072 || all params: 1,617,573,376 || trainable%: 4.5660


Load Teacher Model in 4-bit

In [256]:
teacher = AutoModelForCausalLM.from_pretrained(
    TEACHER_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
)

teacher.eval()
teacher.config.use_cache = True

print("Teacher loaded.")

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


[GPU] 31, 21450, 81920


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


[GPU] 32, 21450, 81920
Teacher loaded.


Helper: Free Memory

In [257]:
def cleanup():
    gc.collect()
    torch.cuda.empty_cache()

Helper: Generate Student Completions

In [258]:
@torch.no_grad()
def generate_group_batched(prompt_batch, group_size=GROUP_SIZE):
    expanded_prompts = []

    for prompt in prompt_batch:
        expanded_prompts.extend([prompt] * group_size)

    inputs = tokenizer(
        expanded_prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=1024,
    ).to(DEVICE)

    prompt_len = inputs["input_ids"].shape[1]

    outputs = student.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=True,
        temperature=TEMPERATURE,
        top_p=TOP_P,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    completions = outputs[:, prompt_len:]

    return inputs["input_ids"], outputs, completions, len(prompt_batch)

Helper: Compute Token Logprobs

In [259]:
def get_completion_logprobs(model, full_ids, prompt_length):
    """
    Returns logprobs for completion tokens only.

    full_ids shape:
      [batch, total_seq_len]

    output:
      token_logprobs shape [batch, completion_len]
    """

    attention_mask = full_ids.ne(tokenizer.pad_token_id).long()

    outputs = model(
        input_ids=full_ids,
        attention_mask=attention_mask,
    )

    logits = outputs.logits

    # logits at position t predict token t+1
    shifted_logits = logits[:, :-1, :]
    shifted_labels = full_ids[:, 1:]

    logprobs = F.log_softmax(shifted_logits, dim=-1)
    token_logprobs = torch.gather(
        logprobs,
        dim=-1,
        index=shifted_labels.unsqueeze(-1),
    ).squeeze(-1)

    # completion starts after prompt_length
    # because shifted_labels is offset by 1, completion token index starts at prompt_length - 1
    completion_logprobs = token_logprobs[:, prompt_length - 1:]

    return completion_logprobs

Helper: Completion Mask

In [260]:
def make_completion_mask(completions):
    """
    1 for real completion tokens, 0 after padding/eos.
    """
    mask = completions.ne(tokenizer.pad_token_id).float()

    # Optional: stop counting after eos
    eos = tokenizer.eos_token_id
    for i in range(completions.shape[0]):
        eos_positions = (completions[i] == eos).nonzero(as_tuple=True)[0]
        if len(eos_positions) > 0:
            first_eos = eos_positions[0].item()
            mask[i, first_eos + 1:] = 0.0

    return mask

Helper: One On-Policy Distillation Step

In [261]:
optimizer = torch.optim.AdamW(student.parameters(), lr=LR)

def opd_train_step_batched(prompt_batch):
    student.eval()

    batch_size = len(prompt_batch)

    # 1. Student samples completions for multiple prompts at once
    with torch.no_grad():
        prompt_ids, full_ids, completions, actual_batch_size = generate_group_batched(
            prompt_batch,
            GROUP_SIZE,
        )

        prompt_length = prompt_ids.shape[1]

        # old student logprobs: no gradient
        old_student_logprobs = get_completion_logprobs(
            student,
            full_ids,
            prompt_length,
        ).detach()

        # teacher logprobs: no gradient
        teacher_logprobs = get_completion_logprobs(
            teacher,
            full_ids,
            prompt_length,
        ).detach()

        mask = make_completion_mask(completions).to(old_student_logprobs.device)

        min_len = min(
            old_student_logprobs.shape[1],
            teacher_logprobs.shape[1],
            mask.shape[1],
        )

        old_student_logprobs = old_student_logprobs[:, :min_len]
        teacher_logprobs = teacher_logprobs[:, :min_len]
        mask = mask[:, :min_len]

        # reward = teacher_logprob - old_student_logprob
        token_rewards = teacher_logprobs - old_student_logprobs

        # sequence-level reward
        seq_rewards = (token_rewards * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)

        # Reshape:
        # [batch_size * group_size] -> [batch_size, group_size]
        seq_rewards_grouped = seq_rewards.view(batch_size, GROUP_SIZE)

        # Normalize advantages within each prompt group
        group_means = seq_rewards_grouped.mean(dim=1, keepdim=True)
        group_stds = seq_rewards_grouped.std(dim=1, keepdim=True)

        advantages_grouped = torch.where(
            group_stds > 1e-6,
            (seq_rewards_grouped - group_means) / (group_stds + 1e-6),
            seq_rewards_grouped - group_means,
        )

        advantages = advantages_grouped.reshape(-1).detach()

    student.train()

    # 2. Current student logprobs with gradient
    current_logprobs = get_completion_logprobs(
        student,
        full_ids,
        prompt_length,
    )

    current_logprobs = current_logprobs[:, :min_len]

    # 3. Policy-gradient-style loss
    per_sequence_logprob = (current_logprobs * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)

    loss = -(advantages * per_sequence_logprob).mean()

    loss = loss / ACCUM_STEPS
    loss.backward()

    return {
        "loss": loss.item() * ACCUM_STEPS,
        "mean_reward": seq_rewards.mean().item(),
        "std_reward": seq_rewards.std().item(),
        "mean_advantage": advantages.mean().item(),
        "num_prompts": batch_size,
        "num_completions": batch_size * GROUP_SIZE,
    }

Smoke Test on One Prompt

In [262]:
# Run cell only for debugging.
# Then restart runtime before real training.
# For real training, skip cell.

# optimizer.zero_grad()

# stats = opd_train_step_batched(prompts[:PROMPTS_PER_STEP])

# torch.nn.utils.clip_grad_norm_(student.parameters(), 1.0)
# optimizer.step()
# optimizer.zero_grad()

# stats

Train on HumanEval Prompts

In [263]:
logs = []
global_step = 0

optimizer.zero_grad()

for epoch in range(EPOCHS):
    order = list(range(len(prompts)))
    random.shuffle(order)

    prompt_batches = [
        order[i : i + PROMPTS_PER_STEP]
        for i in range(0, len(order), PROMPTS_PER_STEP)
    ]

    pbar = tqdm(prompt_batches, desc=f"Epoch {epoch+1}")

    for step, batch_indices in enumerate(pbar):
        prompt_batch = [prompts[i] for i in batch_indices]
        task_id_batch = [task_ids[i] for i in batch_indices]

        stats = opd_train_step_batched(prompt_batch)

        stats["epoch"] = epoch
        stats["step"] = step
        stats["task_ids"] = task_id_batch
        logs.append(stats)

        if (step + 1) % ACCUM_STEPS == 0:
            torch.nn.utils.clip_grad_norm_(student.parameters(), 1.0)
            optimizer.step()
            optimizer.zero_grad()
            global_step += 1

        if step % 5 == 0:
            pbar.set_postfix({
                "loss": round(stats["loss"], 4),
                "reward": round(stats["mean_reward"], 4),
                "prompts": stats["num_prompts"],
                "comps": stats["num_completions"],
            })

        if step > 0 and step % CHECKPOINT_EVERY == 0:
            ckpt_dir = f"/content/opd_checkpoint_batch_step_{step}"
            student.save_pretrained(ckpt_dir)
            tokenizer.save_pretrained(ckpt_dir)
            print("Saved checkpoint:", ckpt_dir)

        if step % 10 == 0:
            cleanup()

# final optimizer step if leftovers exist
if (step + 1) % ACCUM_STEPS != 0:
    torch.nn.utils.clip_grad_norm_(student.parameters(), 1.0)
    optimizer.step()
    optimizer.zero_grad()

print("Training complete.")

Epoch 1:   0%|          | 0/41 [00:00<?, ?it/s]

[GPU] 15, 21454, 81920
[GPU] 20, 21454, 81920
[GPU] 22, 21454, 81920
[GPU] 99, 32540, 81920


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


[GPU] 15, 21396, 81920
[GPU] 15, 21396, 81920
[GPU] 19, 21396, 81920
[GPU] 19, 21396, 81920
[GPU] 19, 21396, 81920
[GPU] 19, 21396, 81920


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


...

[GPU] 15, 50574, 81920
[GPU] 19, 50574, 81920
[GPU] 19, 50574, 81920


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


[GPU] 16, 50574, 81920
[GPU] 15, 50574, 81920
[GPU] 18, 50574, 81920
[GPU] 19, 50574, 81920
[GPU] 19, 50574, 81920
[GPU] 19, 50574, 81920
[GPU] 19, 50574, 81920
[GPU] 20, 50574, 81920


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Training complete.


Save LoRA Adapter

In [264]:
MODEL_DIR = os.path.join(SAVE_DIR, "lora_adapter")

student.save_pretrained(MODEL_DIR)
tokenizer.save_pretrained(MODEL_DIR)

print("Saved model to:", MODEL_DIR)

Saved model to: /content/drive/MyDrive/algoverse_opd_experiment/opd_seed789_grp4_pps4_tok128/lora_adapter


Inspect Training Logs

In [265]:
import pandas as pd

df_logs = pd.DataFrame(logs)
df_logs.tail()

,loss,mean_reward,std_reward,mean_advantage,num_prompts,num_completions,epoch,step,task_ids
36,-0.020702,-0.182814,0.075729,1.490116e-08,4,16,0,36,"[HumanEval/68, HumanEval/49, HumanEval/92, Hum..."
37,-0.003045,-0.183354,0.038131,-7.450581e-08,4,16,0,37,"[HumanEval/76, HumanEval/91, HumanEval/77, Hum..."
38,-0.033579,-0.175805,0.063458,3.725290e-09,4,16,0,38,"[HumanEval/52, HumanEval/133, HumanEval/148, H..."
39,-0.023279,-0.180501,0.082891,0.000000e+00,4,16,0,39,"[HumanEval/134, HumanEval/79, HumanEval/136, H..."
40,-0.004272,-0.163608,0.052063,-1.788139e-07,4,16,0,40,"[HumanEval/34, HumanEval/125, HumanEval/9, Hum..."


In [266]:
TRAINING_LOG_PATH = os.path.join(SAVE_DIR, "training_log.csv")
df_logs.to_csv(TRAINING_LOG_PATH, index=False)

print("Saved training log:", TRAINING_LOG_PATH)

Saved training log: /content/drive/MyDrive/algoverse_opd_experiment/opd_seed789_grp4_pps4_tok128/training_log.csv


In [267]:
df_logs[["loss", "mean_reward", "std_reward"]].describe()

,loss,mean_reward,std_reward
count,41.000000,41.000000,41.000000
mean,-0.038286,-0.232205,0.108834
std,0.048368,0.109717,0.113155
min,-0.207444,-0.747231,0.036727
25%,-0.052065,-0.236533,0.060007
50%,-0.026881,-0.200230,0.075729
75%,-0.011742,-0.180501,0.096903
max,0.044864,-0.143225,0.658048


Generate One Sample After Training

In [268]:
# student.eval()

# test_prompt = prompts[0]

# inputs = tokenizer(test_prompt, return_tensors="pt").to(DEVICE)

# with torch.no_grad():
#     out = student.generate(
#         **inputs,
#         max_new_tokens=256,
#         do_sample=False,
#         pad_token_id=tokenizer.pad_token_id,
#         eos_token_id=tokenizer.eos_token_id,
#     )

# completion = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
# print(completion)

Evaluation Helpers

In [269]:
import signal
import contextlib
import traceback

class TimeoutException(Exception):
    pass

def timeout_handler(signum, frame):
    raise TimeoutException("Timed out")

@contextlib.contextmanager
def time_limit(seconds):
    old_handler = signal.signal(signal.SIGALRM, timeout_handler)
    signal.alarm(seconds)
    try:
        yield
    finally:
        signal.alarm(0)
        signal.signal(signal.SIGALRM, old_handler)

def run_humaneval_test(problem, completion, timeout=3):
    code = problem["prompt"] + completion + "\n" + problem["test"] + "\ncheck(" + problem["entry_point"] + ")"

    local_env = {}

    try:
        with time_limit(timeout):
            exec(code, local_env)
        return True, None

    except Exception as e:
        return False, repr(e)

Generate Deterministic HumanEval completions

In [270]:
def generate_trained_completions_batch(problems, batch_size=8):
    prompt_list = [make_prompt(p) for p in problems]

    inputs = tokenizer(
        prompt_list,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=1024,
    ).to(DEVICE)

    prompt_len = inputs["input_ids"].shape[1]

    with torch.no_grad():
        out = student.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    completions = []
    for i in range(out.shape[0]):
        completion = tokenizer.decode(
            out[i][prompt_len:],
            skip_special_tokens=True,
        )
        completions.append(completion)

    return completions

Clean Generated Code

In [271]:
def clean_completion(text):
    # Remove markdown fences but preserve indentation
    text = text.replace("```python", "")
    text = text.replace("```", "")

    # Stop when model starts adding tests/explanations
    stop_markers = [
        "\n# Test",
        "\n# test",
        "\nassert ",
        "\ndef check",
        "\n# Check",
        "\nExplanation:",
        "\nThe function",
        "\nThis function",
    ]

    for marker in stop_markers:
        if marker in text:
            text = text.split(marker)[0]

    # IMPORTANT:
    # use rstrip(), not strip()
    # strip() removes leading spaces and destroys function-body indentation
    return text.rstrip() + "\n"

Run HumanEval

In [272]:
results = []
EVAL_BATCH_SIZE = 8

student.eval()

for start in tqdm(range(0, len(dataset), EVAL_BATCH_SIZE), desc="Evaluating HumanEval"):
    batch = dataset.select(range(start, min(start + EVAL_BATCH_SIZE, len(dataset))))
    raw_completions = generate_trained_completions_batch(batch, batch_size=EVAL_BATCH_SIZE)

    for problem, raw_completion in zip(batch, raw_completions):
        completion = clean_completion(raw_completion)
        passed, error = run_humaneval_test(problem, completion)

        results.append({
            "task_id": problem["task_id"],
            "passed": passed,
            "completion": completion,
            "error": error,
        })

passed = sum(r["passed"] for r in results)
total = len(results)
pass_at_1 = passed / total

print(f"HumanEval pass@1: {pass_at_1:.3f} ({passed}/{total})")

Evaluating HumanEval:   0%|          | 0/21 [00:00<?, ?it/s]

[GPU] 14, 21396, 81920
[GPU] 16, 21396, 81920
[GPU] 16, 21396, 81920
[GPU] 16, 21396, 81920
[GPU] 16, 21396, 81920
[GPU] 16, 21396, 81920
[GPU] 15, 21396, 81920
[GPU] 16, 21396, 81920
[GPU] 17, 21396, 81920
[GPU] 14, 21396, 81920
[GPU] 17, 21396, 81920
[GPU] 17, 21396, 81920
[GPU] 16, 21396, 81920
[GPU] 16, 21396, 81920
[GPU] 17, 21396, 81920
[GPU] 18, 21396, 81920
[GPU] 17, 21396, 81920
[GPU] 17, 21396, 81920
[GPU] 17, 21396, 81920
[GPU] 17, 21396, 81920
[GPU] 17, 21396, 81920
[GPU] 14, 21396, 81920
[GPU] 17, 21396, 81920
[GPU] 17, 21396, 81920
[GPU] 0, 21396, 81920
[GPU] 0, 21396, 81920
[GPU] 15, 21396, 81920
[GPU] 17, 21396, 81920
[GPU] 17, 21396, 81920
[GPU] 17, 21396, 81920
[GPU] 17, 21396, 81920
[GPU] 17, 21396, 81920
[GPU] 17, 21396, 81920
[GPU] 17, 21396, 81920
[GPU] 17, 21396, 81920
[GPU] 17, 21396, 81920
[GPU] 17, 21396, 81920
[GPU] 17, 21396, 81920
[GPU] 15, 21396, 81920
[GPU] 17, 21396, 81920
[GPU] 15, 21396, 81920
[GPU] 14, 21396, 81920
[GPU] 17, 21396, 81920
[GPU] 18, 213

Save HumanEval Results

In [273]:
RESULTS_PATH = os.path.join(SAVE_DIR, "humaneval_results.json")

with open(RESULTS_PATH, "w") as f:
    json.dump(results, f, indent=2)

print("Saved:", RESULTS_PATH)

Saved: /content/drive/MyDrive/algoverse_opd_experiment/opd_seed789_grp4_pps4_tok128/humaneval_results.json


OPD Results & Load Base Model

In [274]:
OPD_PASSED = passed
OPD_TOTAL = total
OPD_PASS_AT_1 = pass_at_1

import gc
import torch

gc.collect()
torch.cuda.empty_cache()

base_student = AutoModelForCausalLM.from_pretrained(
    STUDENT_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
)

base_student.eval()

print("Saved OPD result:", OPD_PASS_AT_1, OPD_PASSED, OPD_TOTAL)
print("Base model loaded.")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Saved OPD result: 0.5914634146341463 97 164
Base model loaded.


Run Baseline

In [275]:
def generate_base_completions_batch(problems, batch_size=8):
    prompt_list = [make_prompt(p) for p in problems]

    inputs = tokenizer(
        prompt_list,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=1024,
    ).to(DEVICE)

    prompt_len = inputs["input_ids"].shape[1]

    with torch.no_grad():
        out = base_student.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    completions = []
    for i in range(out.shape[0]):
        completion = tokenizer.decode(
            out[i][prompt_len:],
            skip_special_tokens=True,
        )
        completions.append(completion)

    return completions

In [276]:
results = []
EVAL_BATCH_SIZE = 8

base_student.eval()

for start in tqdm(range(0, len(dataset), EVAL_BATCH_SIZE), desc="Evaluating HumanEval"):
    batch = dataset.select(range(start, min(start + EVAL_BATCH_SIZE, len(dataset))))
    raw_completions = generate_base_completions_batch(batch, batch_size=EVAL_BATCH_SIZE)

    for problem, raw_completion in zip(batch, raw_completions):
        completion = clean_completion(raw_completion)
        passed, error = run_humaneval_test(problem, completion)

        results.append({
            "task_id": problem["task_id"],
            "passed": passed,
            "completion": completion,
            "error": error,
        })

passed = sum(r["passed"] for r in results)
total = len(results)
pass_at_1 = passed / total

print(f"HumanEval pass@1: {pass_at_1:.3f} ({passed}/{total})")

Evaluating HumanEval:   0%|          | 0/21 [00:00<?, ?it/s]

[GPU] 17, 21396, 81920
[GPU] 17, 21406, 81920
[GPU] 17, 21406, 81920
[GPU] 18, 21406, 81920
[GPU] 18, 21406, 81920
[GPU] 18, 21406, 81920
[GPU] 17, 21406, 81920
[GPU] 17, 21406, 81920
[GPU] 18, 21406, 81920
[GPU] 18, 21406, 81920
[GPU] 18, 21406, 81920
[GPU] 18, 21406, 81920
[GPU] 18, 21406, 81920
[GPU] 17, 21406, 81920
[GPU] 18, 21406, 81920
[GPU] 18, 21406, 81920
[GPU] 17, 21406, 81920
[GPU] 17, 21406, 81920
[GPU] 17, 21406, 81920
[GPU] 17, 21406, 81920
[GPU] 18, 21406, 81920
[GPU] 18, 21406, 81920
[GPU] 18, 21406, 81920
[GPU] 17, 21406, 81920
[GPU] 17, 21406, 81920
[GPU] 19, 21406, 81920
[GPU] 19, 21406, 81920
[GPU] 19, 21406, 81920
[GPU] 19, 21406, 81920
[GPU] 17, 21406, 81920
[GPU] 18, 21406, 81920
[GPU] 19, 21406, 81920
[GPU] 19, 21406, 81920
[GPU] 17, 21406, 81920
[GPU] 17, 21406, 81920
[GPU] 18, 21406, 81920
[GPU] 18, 21406, 81920
[GPU] 19, 21406, 81920
[GPU] 19, 21406, 81920
[GPU] 19, 21406, 81920
[GPU] 17, 21406, 81920
[GPU] 19, 21406, 81920
[GPU] 19, 21406, 81920
[GPU] 19, 2

Compare Against Baseline

In [277]:
BASE_PASSED = passed
BASE_TOTAL = total
BASE_PASS_AT_1 = pass_at_1

print(f"Base rerun: {BASE_PASS_AT_1:.3f} ({BASE_PASSED}/{BASE_TOTAL})")

Base rerun: 0.433 (71/164)


In [278]:
print(f"Base rerun: {BASE_PASS_AT_1:.3f} ({BASE_PASSED}/{BASE_TOTAL})")
print(f"OPD:        {OPD_PASS_AT_1:.3f} ({OPD_PASSED}/{OPD_TOTAL})")
print(f"Delta:      {OPD_PASS_AT_1 - BASE_PASS_AT_1:+.3f}")

Base rerun: 0.433 (71/164)
OPD:        0.591 (97/164)
Delta:      +0.159


Save Config & Metrics

In [279]:
import json

config = {
    "student_model": STUDENT_MODEL,
    "teacher_model": TEACHER_MODEL,
    "group_size": GROUP_SIZE,
    "prompts_per_step": PROMPTS_PER_STEP,
    "max_new_tokens": MAX_NEW_TOKENS,
    "temperature": TEMPERATURE,
    "top_p": TOP_P,
    "lora_rank": LORA_RANK,
    "lora_alpha": LORA_ALPHA,
    "learning_rate": LR,
    "epochs": EPOCHS,
    "accum_steps": ACCUM_STEPS,
    "seed": SEED,
}

with open(os.path.join(SAVE_DIR, "config.json"), "w") as f:
    json.dump(config, f, indent=2)

print("Saved config.")

Saved config.


In [280]:
final_train_log = logs[-1]

metrics = {
    "base_passed": BASE_PASSED,
    "base_total": BASE_TOTAL,
    "base_pass_at_1": BASE_PASS_AT_1,

    "opd_passed": OPD_PASSED,
    "opd_total": OPD_TOTAL,
    "opd_pass_at_1": OPD_PASS_AT_1,

    "delta": OPD_PASS_AT_1 - BASE_PASS_AT_1,

    "final_loss": final_train_log["loss"],
    "final_mean_reward": final_train_log["mean_reward"],
    "final_std_reward": final_train_log["std_reward"],
    "final_mean_advantage": final_train_log["mean_advantage"],
}

with open(os.path.join(SAVE_DIR, "metrics.json"), "w") as f:
    json.dump(metrics, f, indent=2)

print("Saved metrics.")
print(metrics)

Saved metrics.
{'base_passed': 71, 'base_total': 164, 'base_pass_at_1': 0.4329268292682927, 'opd_passed': 97, 'opd_total': 164, 'opd_pass_at_1': 0.5914634146341463, 'delta': 0.15853658536585363, 'final_loss': -0.004272060468792915, 'final_mean_reward': -0.16360782086849213, 'final_std_reward': 0.052063312381505966, 'final_mean_advantage': -1.7881393432617188e-07}


Debug

In [281]:

failed = [r for r in results if not r["passed"]]

for r in failed[:5]:
    print("=" * 80)
    print(r["task_id"])
    print(r["error"])
    print(r["completion"][:1500])

HumanEval/1
SyntaxError('unterminated triple-quoted string literal (detected at line 57)', ('<string>', 35, 5, '    """ Input to this function is a string containing multiple groups of nested parentheses. Your goal is to', 35, 5))
    result = []
    current_group = ''
    depth = 0

    for char in paren_string.replace(' ', ''):

        if char == '(':
            depth += 1
            current_group += char
        elif char == ')':
            depth -= 1
            current_group += char
            if depth == 0:
                result.append(current_group)
                current_group = ''

    return result


from typing import List


def separate_paren_groups(paren_string: str) -> List[str]:
    """ Input to this function is a string containing multiple groups of nested parentheses. Your goal is to

HumanEval/5
AssertionError()
    return numbers if len(numbers) == 0 else [numbers[0]] + [delimeter] + intersperse(numbers[1:], delimeter)

HumanEval/6
AssertionError()
    return 

In [282]:
problem = dataset[0]
raw = generate_trained_completions_batch([problem], batch_size=1)[0]

print("RAW COMPLETION:")
print(raw[:2000])

print("\nCLEANED COMPLETION:")
print(clean_completion(raw)[:2000])

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


[GPU] 14, 21406, 81920


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


[GPU] 14, 21406, 81920


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


...

[GPU] 14, 21406, 81920


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


[GPU] 14, 21406, 81920


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


[GPU] 14, 21406, 81920
RAW COMPLETION:
    numbers.sort()
    for i in range(1, len(numbers)):
        if abs(numbers[i] - numbers[i - 1]) < threshold:
            return True
    return False

# Function to check the correctness of the solution
def check_function():
    assert has_close_elements([1.0, 2.0, 3.0], 0.5) == False, "Test case 1 failed"
    assert has_close_elements([1.0, 2.8, 3.0, 4.0, 5.0, 2.0], 0.3)

CLEANED COMPLETION:
    numbers.sort()
    for i in range(1, len(numbers)):
        if abs(numbers[i] - numbers[i - 1]) < threshold:
            return True
    return False

# Function to check the correctness of the solution

